In [65]:
import numpy as np
import pandas as pd

In [66]:
xls = pd.ExcelFile("Data.xlsx")
print(xls.sheet_names)

['Fund Info', 'Fund Returns', 'Factor Returns']


In [67]:
info = pd.read_excel(xls, sheet_name="Fund Info")
fund = pd.read_excel(xls, sheet_name="Fund Returns")
fac  = pd.read_excel(xls, sheet_name="Factor Returns")



In [68]:
print(fac.shape)
print(fac.size)
print(fac.dtypes)
print(fac.info())


(19017, 3)
57051
date            datetime64[us]
total_return           float64
index_ticker               str
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 19017 entries, 0 to 19016
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   date          19017 non-null  datetime64[us]
 1   total_return  19017 non-null  float64       
 2   index_ticker  19017 non-null  str           
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 445.8 KB
None


In [69]:
fund['date'] = pd.to_datetime(fund['date'])
fac['date']  = pd.to_datetime(fac['date'])
ret_wide = fund.pivot(index='date', columns='ticker', values='total_return').sort_index()
fac_wide = fac.pivot(index='date', columns='index_ticker', values='total_return').sort_index()

In [70]:
fac_wide.shape

(6339, 3)

In [71]:
sub = ret_wide[['SPY','AGG']].dropna()
sub5 = ret_wide.dropna()

sub.shape

(5702, 2)

In [72]:
info['dividend_yield'] = info['dividend_yield'].fillna(0)

info

,ticker,fund_name,dividend_yield
0,IEFA,iShares Core MSCI EAFE ETF,0.03278
1,GLD,SPDR Gold Shares,0.00000
2,AGG,iShares Core US Aggregate Bond ETF,0.03974
3,VEA,Vanguard Developed Markets Index Fund;ETF,0.02034
4,SPY,State Street SPDR S&P 500 ETF Trust,0.00987


In [73]:
w = [0.6, 0.3, 0.1]
rp = ret_wide[['SPY','AGG','GLD']].dropna().values @ w


rp

array([ 0.000927, -0.006383,  0.00371 , ...,  0.001904,  0.004879,
       -0.00119 ], shape=(5413,))

**Portfolio returns**

In [74]:
def portfolio_volatility(w, cov):
    return float(np.sqrt(w.T @ cov @ w))


def sharpe_ratio(w, mean, cov):
    vol = portfolio_volatility(w, cov)
    return float(w @ mean) / vol if vol else 0.0


def max_drawdown(pr):
    cum = np.cumprod(1 + pr)
    peak = np.maximum.accumulate(cum)
    return float(np.min((cum - peak) / peak))

**Optimization Engine**

In [75]:
from scipy.optimize import minimize

def optimize_weights(cols, ret_wide, objective, min_w=0.0, max_w=1.0,
                     min_dividend=None, dividends=None, maxiter=100):
    n = len(cols)
    if n == 1:
        return np.array([1.0]), None, None, True
    m = ret_wide[cols].dropna().values
    mean = np.mean(m, axis=0)
    cov = np.cov(m, rowvar=False)
    divs = np.array([dividends[t] for t in cols]) if min_dividend is not None else None # type: ignore
    bounds, constraints = _build_constraints(n, divs, min_w, max_w, min_dividend)
    res = minimize(
        objective,
        np.array([1 / n] * n),
        args=(mean, cov),
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": maxiter},
    )
    if not res.success:
        raise ValueError(f"Optimization failed: {res.message}")
    _check_weights(res.x)
    return res.x, mean, cov, res.success


In [76]:


def min_vol_weights(cols, ret_wide, **kwargs):
    return optimize_weights(cols, ret_wide, lambda w, mean, cov: portfolio_volatility(w, cov), **kwargs)


def max_sharpe_weights(cols, ret_wide, **kwargs):
    return optimize_weights(cols, ret_wide, lambda w, mean, cov: -sharpe_ratio(w, mean, cov), **kwargs)



In [77]:

def equal_weights(n):
    return np.array([1 / n] * n)

**Risk Parity + Drawdown (advanced strategies)**

In [78]:
def risk_parity_weights(cols, ret_wide, **kwargs):
    # Normalized: raw sum((rc - var/N)^2) is ~1e-10 scale, SLSQP stalls at init.
    def objective(w, mean, cov):
        var = max(float(w.T @ cov @ w), 1e-12)
        rc = w * (cov @ w)
        return float(np.sum((rc / var - 1 / len(w)) ** 2))

    return optimize_weights(cols, ret_wide, objective, **kwargs)


def min_drawdown_weights(cols, ret_wide, maxiter=300, min_w=0.0, max_w=1.0,
                         min_dividend=None, dividends=None):
    n = len(cols)
    if n == 1:
        return np.array([1.0]), True
    m = ret_wide[cols].dropna().values
    divs = np.array([dividends[t] for t in cols]) if min_dividend is not None else None
    bounds, constraints = _build_constraints(n, divs, min_w, max_w, min_dividend)
    res = minimize(
        lambda w: abs(max_drawdown(m @ w)),
        np.array([1 / n] * n),
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": maxiter},
    )
    if not res.success:
        raise ValueError(f"Optimization failed: {res.message}")
    _check_weights(res.x)
    return res.x, res.success

**Constraints**

In [79]:
def _build_constraints(n, dividends, min_w=0.0, max_w=1.0, min_dividend=None):
    bounds = [(min_w, max_w)] * n
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]
    if min_dividend is not None:
        constraints.append({"type": "ineq", "fun": lambda w: float(w @ dividends) - min_dividend})
    return bounds, constraints



In [80]:

def _check_weights(w):
    if abs(float(np.sum(w)) - 1) > 1e-6 or bool((w < -1e-9).any()):
        raise ValueError("Invalid weights: must sum to 1 and never be negative")

In [81]:

w, mean, cov, ok = risk_parity_weights(['VEA','AGG'], ret_wide)
print(dict(zip(['VEA','AGG'], np.round(w,4))), ok)
w2, ok2 = min_drawdown_weights(['SPY','AGG','GLD'], ret_wide)

print(dict(zip(['SPY','AGG','GLD'], np.round(w2,4))), ok2)



{'VEA': np.float64(0.2013), 'AGG': np.float64(0.7987)} True
{'SPY': np.float64(0.1181), 'AGG': np.float64(0.5643), 'GLD': np.float64(0.3176)} True


In [82]:

w, mean, cov, ok = min_vol_weights(['SPY','AGG','GLD'], ret_wide)
print(dict(zip(['SPY','AGG','GLD'], np.round(w,4))), ok, round(float(w.sum()),6))
w2, m2, c2, ok2 = max_sharpe_weights(['IEFA','GLD','AGG','VEA','SPY'], ret_wide)


print(dict(zip(['IEFA','GLD','AGG','VEA','SPY'], np.round(w2,4))), ok2)

{'SPY': np.float64(0.0666), 'AGG': np.float64(0.9177), 'GLD': np.float64(0.0157)} True 1.0
{'IEFA': np.float64(0.0), 'GLD': np.float64(0.2005), 'AGG': np.float64(0.3445), 'VEA': np.float64(0.0), 'SPY': np.float64(0.455)} True


In [91]:
y = dict(zip(info['ticker'], info['dividend_yield']))

tickers = ['IEFA', 'GLD', 'AGG', 'VEA', 'SPY']
web = {'IEFA':10.65, 'GLD':5.02, 'AGG':39.99, 'VEA':5.01, 'SPY':39.33}

w5, m5, c5, ok5 = max_sharpe_weights(
    tickers, ret_wide, min_w=0.05, max_w=0.40,
    min_dividend=0.025, dividends=y)

print('success:', ok5, '| sum:', round(float(w5.sum()), 6),
      '| yield:', round(float(w5 @ np.array([y[t] for t in tickers])), 6))
for t, o in zip(tickers, w5 * 100):
    print(t, 'ours', round(float(o), 2), 'web', web[t], 'diff', round(float(o) - web[t], 2))

success: True | sum: 1.0 | yield: 0.025
IEFA ours 14.71 web 10.65 diff 4.06
GLD ours 7.2 web 5.02 diff 2.18
AGG ours 40.0 web 39.99 diff 0.01
VEA ours 5.0 web 5.01 diff -0.01
SPY ours 33.1 web 39.33 diff -6.23
